# BSE Corporate Announcements PDF Scraper
**ArthLLM Training Corpus**

| Category | BSE Filter | Est. Files (2011-2026) |
|----------|-----------|------------------------|
| Board Meeting | `Board Meeting` | ~180,000 |
| Insider Trading | `Insider Trading/SAST` | ~250,000 |
| Dividend | `Dividend` | ~30,000 |
| Bonus / Split | `Sub-Division/Splits/Consolidation` | ~2,000 |
| Buyback | `Buy Back` | ~1,200 |
| Demerger | `Scheme of Arrangement/Amalgamation` | ~5,000 |
| **TOTAL** | | **~468,000 PDFs (~94 GB)** |

## Cell 1 -- Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/ArthLLM-2B/corporate_announcements'
os.makedirs(f'{BASE}/logs', exist_ok=True)
for cat in ['board_meetings','insider_trading','dividend','bonus_splits','buyback','demerger']:
    os.makedirs(f'{BASE}/{cat}', exist_ok=True)
print(f'Base: {BASE}')
print('Folders ready.')

## Cell 2 -- Install Dependencies

In [ ]:
%pip install requests tqdm pdfplumber pymupdf -q
print('Done.')

## Cell 3 -- Scraper Engine

In [ ]:
import requests, os, json, time, random, hashlib, csv
from datetime import datetime, timedelta
from tqdm.notebook import tqdm

LOGS = f'{BASE}/logs'

CATEGORIES = {
    'board_meetings':  'Board Meeting',
    'insider_trading': 'Insider Trading/SAST',
    'dividend':        'Dividend',
    'bonus_splits':    'Sub-Division/Splits/Consolidation',
    'buyback':         'Buy Back',
    'demerger':        'Scheme of Arrangement/Amalgamation',
}

RATE_LIMIT     = 3
PAUSE_EVERY    = 100
PAUSE_SECS     = 30
RETRY_WAIT     = 60
SESSION_BUDGET = 11 * 3600  # 11 hours -- stop before Colab's 12hr kill
MIN_FILE_SIZE  = 30720
WINDOW_DAYS    = 15

BSE_API  = 'https://api.bseindia.com/BseIndiaAPI/api/AnnGetData/w'
PDF_LIVE = 'https://www.bseindia.com/xml-data/corpfiling/AttachLive/{}'
PDF_HIST = 'https://www.bseindia.com/xml-data/corpfiling/AttachHis/{}'

START = datetime(2011, 1, 1)
END   = datetime(2026, 5, 8)


class BSE:
    def __init__(self):
        self.s = requests.Session()
        self.s.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                          '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Referer': 'https://www.bseindia.com/corporates/ann.html',
            'Origin': 'https://www.bseindia.com',
        })
        self.last = 0
        self.dl_count = 0
        self._warm()

    def _warm(self):
        try:
            print('  Warming BSE session...', flush=True)
            self.s.get('https://www.bseindia.com', timeout=15)
            time.sleep(1)
            self.s.get('https://www.bseindia.com/corporates/ann.html', timeout=15)
            print('  Session ready.', flush=True)
        except Exception as e:
            print(f'  Warm-up warning: {e}', flush=True)

    def _wait(self):
        gap = time.time() - self.last
        if gap < RATE_LIMIT:
            time.sleep(RATE_LIMIT - gap + random.uniform(0.2, 0.8))
        self.last = time.time()

    def _pause(self):
        self.dl_count += 1
        if self.dl_count % PAUSE_EVERY == 0:
            print(f'  Cooling {PAUSE_SECS}s after {self.dl_count} downloads...', flush=True)
            time.sleep(PAUSE_SECS)

    def api(self, params):
        self._wait()
        r = self.s.get(BSE_API, params=params, timeout=60)
        if r.status_code in (429, 503):
            print(f'  Rate limited ({r.status_code}), waiting 60s...', flush=True)
            time.sleep(60)
            r = self.s.get(BSE_API, params=params, timeout=60)
        r.raise_for_status()
        return r.json()

    def get_pdf(self, url):
        self._wait()
        r = self.s.get(url, timeout=120)
        if r.status_code == 403:
            self._warm()
            time.sleep(5)
            self._wait()
            r = self.s.get(url, timeout=120)
        r.raise_for_status()
        return r.content


class Logger:
    def __init__(self):
        self._dl = f'{LOGS}/download_log.csv'
        self._fail = f'{LOGS}/failed_downloads.log'
        if not os.path.exists(self._dl):
            with open(self._dl, 'w', newline='') as f:
                csv.writer(f).writerow(['timestamp','category','scrip','filename','url','bytes','md5'])
        if not os.path.exists(self._fail):
            with open(self._fail, 'w') as f:
                f.write(f'# Failed Downloads -- {datetime.now()}\n')
        self.hashes = set()
        try:
            with open(self._dl) as f:
                for row in csv.DictReader(f):
                    if row.get('md5'): self.hashes.add(row['md5'])
        except: pass

    def ok(self, cat, scrip, fname, url, sz, md5):
        with open(self._dl, 'a', newline='') as f:
            csv.writer(f).writerow([datetime.now().isoformat(), cat, scrip, fname, url, sz, md5])
        self.hashes.add(md5)

    def fail(self, cat, url, err):
        with open(self._fail, 'a') as f:
            f.write(f'[{datetime.now().isoformat()}] {cat} | {url} | {err}\n')

    def is_dup(self, md5): return md5 in self.hashes


def load_done_attach(cat):
    p = f'{LOGS}/done_attach_{cat}.txt'
    if os.path.exists(p):
        with open(p) as f:
            return set(line.strip() for line in f if line.strip())
    return set()

def append_done_attach(cat, att):
    with open(f'{LOGS}/done_attach_{cat}.txt', 'a') as f:
        f.write(att + '\n')

def load_prog(cat):
    p = f'{LOGS}/progress_{cat}.json'
    if os.path.exists(p):
        with open(p) as f: return json.load(f)
    return {'done_windows': [], 'total_dl': 0}

def save_prog(cat, d):
    d['updated'] = datetime.now().isoformat()
    with open(f'{LOGS}/progress_{cat}.json', 'w') as f: json.dump(d, f)

def make_windows():
    wins, cur = [], START
    while cur < END:
        w_end = min(cur + timedelta(days=WINDOW_DAYS - 1), END)
        wins.append((cur.strftime('%Y%m%d'), w_end.strftime('%Y%m%d')))
        cur = w_end + timedelta(days=1)
    return wins


def scrape(bse, logger, cat_name, cat_filter, session_start=None):
    out = f'{BASE}/{cat_name}'
    os.makedirs(out, exist_ok=True)
    if session_start is None:
        session_start = time.time()

    prog = load_prog(cat_name)
    done_w = set(prog['done_windows'])
    done_a = load_done_attach(cat_name)
    total = prog['total_dl']

    windows = make_windows()
    pending = [(f,t) for f,t in windows if f'{f}_{t}' not in done_w]

    print(f'\n{"="*60}')
    print(f'  {cat_name} | {cat_filter}')
    print(f'  Windows: {len(windows)} total, {len(pending)} pending')
    print(f'  Resumed: {len(done_a)} attach done, {total} PDFs saved')
    print(f'{"="*60}')

    for wi, (wf, wt) in enumerate(tqdm(pending, desc=cat_name)):
        # Session budget check
        if time.time() - session_start > SESSION_BUDGET:
            print(f'\n  Session budget ({SESSION_BUDGET//3600}h) reached. Re-run to continue.')
            return total

        wdl = 0
        window_ok = False
        total_count = 0
        collected = 0

        page = 1
        while True:
            params = {
                'strCat': cat_filter, 'strPrevDate': wf, 'strScrip': '',
                'strSearch': 'P', 'strToDate': wt, 'strType': 'C',
                'strPageNo': str(page),
            }
            try:
                data = bse.api(params)
                window_ok = True
            except Exception as e:
                logger.fail(cat_name, f'API {wf}-{wt} p{page}', str(e))
                time.sleep(RETRY_WAIT)
                try:
                    data = bse.api(params)
                    window_ok = True
                except Exception as e2:
                    logger.fail(cat_name, f'API RETRY {wf}-{wt} p{page}', str(e2))
                    break

            rows = []
            if isinstance(data, dict) and 'Table' in data: rows = data['Table']
            elif isinstance(data, list): rows = data

            # Grab ROWCNT on page 1
            if page == 1 and isinstance(data, dict) and 'Table1' in data:
                try: total_count = int(data['Table1'][0]['ROWCNT'])
                except: pass

            if not rows:
                break

            collected += len(rows)

            for ann in rows:
                att = (ann.get('ATTACHMENTNAME') or '').strip()
                if not att or att in done_a:
                    continue
                scrip = str(ann.get('SCRIP_CD', ''))
                sp = f'{out}/{att}'

                if os.path.exists(sp):
                    done_a.add(att)
                    append_done_attach(cat_name, att)
                    continue

                content = None
                used_url = None
                for url in [PDF_LIVE.format(att), PDF_HIST.format(att)]:
                    try:
                        content = bse.get_pdf(url)
                        used_url = url
                        break
                    except Exception as e:
                        logger.fail(cat_name, url, str(e))

                if content is None:
                    time.sleep(RETRY_WAIT)
                    try:
                        content = bse.get_pdf(PDF_LIVE.format(att))
                        used_url = PDF_LIVE.format(att)
                    except: pass

                if content is None:
                    continue  # do NOT blacklist

                # HTML check BEFORE size check
                if b'<html' in content[:500].lower() or b'<!doctype' in content[:500].lower():
                    logger.fail(cat_name, att, 'HTML not PDF')
                    done_a.add(att)
                    append_done_attach(cat_name, att)
                    continue

                if len(content) < MIN_FILE_SIZE:
                    done_a.add(att)
                    append_done_attach(cat_name, att)
                    continue

                md5 = hashlib.md5(content).hexdigest()
                if logger.is_dup(md5):
                    done_a.add(att)
                    append_done_attach(cat_name, att)
                    continue

                with open(sp, 'wb') as f: f.write(content)
                logger.ok(cat_name, scrip, att, used_url, len(content), md5)
                done_a.add(att)
                append_done_attach(cat_name, att)
                total += 1; wdl += 1
                bse._pause()

            # ROWCNT stop condition
            if total_count > 0 and collected >= total_count:
                break

            page += 1

        # Only mark done if API succeeded
        if window_ok:
            done_w.add(f'{wf}_{wt}')
            prog['done_windows'] = list(done_w)
            prog['total_dl'] = total
            save_prog(cat_name, prog)

        if (wi+1) % 25 == 0: bse._warm()

    print(f'  DONE: {cat_name} -- {total} PDFs')
    return total

print('Engine loaded.')

## Cell 4 -- Run ALL Categories

In [ ]:
bse = BSE()
logger = Logger()
session_start = time.time()
grand = 0

for name, filt in CATEGORIES.items():
    try:
        grand += scrape(bse, logger, name, filt, session_start=session_start)
    except KeyboardInterrupt:
        print(f'\nInterrupted at {name}. Re-run to resume.')
        break
    except Exception as e:
        print(f'{name} error: {e}')
        import traceback; traceback.print_exc()

print(f'\n{"="*60}')
print(f'TOTAL: {grand:,} PDFs downloaded')
print(f'{"="*60}')

## Cell 5 -- Run Single Category

In [ ]:
TARGET = 'board_meetings'  # board_meetings, insider_trading, dividend, bonus_splits, buyback, demerger

bse = BSE()
logger = Logger()
n = scrape(bse, logger, TARGET, CATEGORIES[TARGET])
print(f'\nDownloaded: {n:,} PDFs')

## Cell 6 -- Check Progress (standalone)

In [ ]:
import os, json
from datetime import datetime, timedelta

_BASE = '/content/drive/MyDrive/ArthLLM-2B/corporate_announcements'
_LOGS = f'{_BASE}/logs'
_CATS = ['board_meetings','insider_trading','dividend','bonus_splits','buyback','demerger']
_START, _END, _WDAYS = datetime(2011,1,1), datetime(2026,5,8), 15

nwins, cur = 0, _START
while cur < _END:
    cur += timedelta(days=_WDAYS)
    nwins += 1

total_pdfs, total_size = 0, 0
print(f'{"Category":20s} {"Windows":>12s} {"Files on disk":>14s} {"Disk Size":>12s}')
print('-'*62)

for cat in _CATS:
    pf = f'{_LOGS}/progress_{cat}.json'
    w = 0
    if os.path.exists(pf):
        with open(pf) as f: w = len(json.load(f).get('done_windows', []))
    cat_dir = f'{_BASE}/{cat}'
    files = [f for f in os.listdir(cat_dir) if os.path.isfile(f'{cat_dir}/{f}')] if os.path.exists(cat_dir) else []
    sz = sum(os.path.getsize(f'{cat_dir}/{f}') for f in files)
    total_pdfs += len(files); total_size += sz
    print(f'{cat:20s} {w:>5d}/{nwins:>5d}   {len(files):>12,d}   {sz/1e9:>8.2f} GB')

print('-'*62)
print(f'{"TOTAL":20s} {"":>12s} {total_pdfs:>12,d}   {total_size/1e9:>8.2f} GB')

## Cell 7 -- Text Extraction (PDF -> TXT)

In [ ]:
import os
import pdfplumber
import fitz  # pymupdf
from tqdm.notebook import tqdm

_BASE = '/content/drive/MyDrive/ArthLLM-2B/corporate_announcements'
_CATS = ['board_meetings','insider_trading','dividend','bonus_splits','buyback','demerger']
MIN_TEXT = 100

def extract_text(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = '\n'.join(p.extract_text() or '' for p in pdf.pages)
            if len(text.strip()) >= MIN_TEXT:
                return text.strip()
    except: pass
    try:
        doc = fitz.open(pdf_path)
        text = '\n'.join(page.get_text() for page in doc)
        doc.close()
        if len(text.strip()) >= MIN_TEXT:
            return text.strip()
    except: pass
    return None


for cat in _CATS:
    cat_dir = f'{_BASE}/{cat}'
    out_file = f'{_BASE}/{cat}.txt'
    done_file = f'{_BASE}/logs/extracted_{cat}.txt'
    if not os.path.exists(cat_dir):
        continue

    already_done = set()
    if os.path.exists(done_file):
        with open(done_file) as f:
            already_done = set(line.strip() for line in f if line.strip())

    pdfs = sorted([f for f in os.listdir(cat_dir) if f.endswith('.pdf')])
    pending = [p for p in pdfs if p not in already_done]
    print(f'\n{cat}: {len(pdfs)} total, {len(pending)} new', flush=True)

    if not pending:
        continue

    extracted, skipped = 0, 0
    with open(out_file, 'a', encoding='utf-8') as out, open(done_file, 'a') as done_f:
        for pdf in tqdm(pending, desc=cat):
            text = extract_text(f'{cat_dir}/{pdf}')
            if text:
                out.write(f'\n\n--- {pdf} ---\n\n')
                out.write(text)
                extracted += 1
            else:
                skipped += 1
            done_f.write(pdf + '\n')

    sz = os.path.getsize(out_file) / 1e6
    print(f'  Extracted: {extracted} | Skipped: {skipped} | Output: {sz:.1f} MB')